# 📄 Step 1 — Data Collection & Vector Store Setup
This notebook downloads dependencies, clones the public `kannada-legal-ai` repository, runs local data collection scripts, and builds the ChromaDB vector store.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted successfully!")

In [ ]:
!pip install -q requests beautifulsoup4 lxml pypdf \
                indic-nlp-library sentencepiece \
                loguru rank-bm25 langdetect \
                chromadb sentence-transformers IndicTransToolkit

print("✅ All data collection dependencies installed!")

In [ ]:
import os
import sys

# Public repo details
GITHUB_USERNAME = 'JhenkarC-555'
REPO_NAME       = 'kannada-legal-ai'
REPO_URL        = f'https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git'

if not os.path.exists(f'/content/{REPO_NAME}'):
    !git clone {REPO_URL}
else:
    print('Repo exists. Pulling latest code...')
    !cd /content/{REPO_NAME} && git pull

%cd /content/{REPO_NAME}
sys.path.insert(0, f'/content/{REPO_NAME}')
print(f'✅ Working directory set to: {os.getcwd()}')

In [ ]:
!python -m data.collection.scrape_india_code

# Verify output file
import json
ipc_path = 'data/raw/ipc_sections/ipc_sections.json'
if os.path.exists(ipc_path):
    with open(ipc_path, encoding='utf-8') as f:
        ipc = json.load(f)
    print(f'✅ IPC sections collected: {len(ipc)}')
else:
    print('⚠️ IPC json file not found.')

In [ ]:
!python -m data.collection.scrape_karnataka_gov

# Verify outputs
ka_path = 'data/raw/karnataka_state_laws/karnataka_laws.json'
vika_path = 'data/raw/legal_aid_pamphlets/vikaspedia_kn.json'

if os.path.exists(ka_path):
    with open(ka_path, encoding='utf-8') as f:
        ka = json.load(f)
    print(f'✅ Karnataka laws collected: {len(ka)}')

if os.path.exists(vika_path):
    with open(vika_path, encoding='utf-8') as f:
        vika = json.load(f)
    print(f'✅ Vikaspedia articles collected: {len(vika)}')

In [ ]:
# Run translation script with colab mode enabled
!python -m data.collection.translate_to_kannada --mode colab

# Check output files in processed data folder
if os.path.exists('data/processed'):
    for file_name in os.listdir('data/processed/'):
        file_path = f'data/processed/{file_name}'
        size = os.path.getsize(file_path)
        print(f'  📄 {file_name} — {size:,} bytes')

In [ ]:
!python -m data.collection.generate_qa_pairs

# Verify dataset split sizes
for split in ['train', 'val', 'test']:
    split_path = f'data/annotated/{split}/qa_pairs.jsonl'
    if os.path.exists(split_path):
        with open(split_path, encoding='utf-8') as f:
            count = sum(1 for line in f if line.strip())
        print(f'✅ {split:10} split : {count} pairs generated')

In [ ]:
!python -m scripts.build_vector_store

# Check created collection
try:
    from rag.vector_store import get_collection_stats
    stats = get_collection_stats()
    print(f"✅ Vector store ready! Total documents: {stats.get('total_documents', 0)}")
except Exception as e:
    print(f"⚠️ Vector store test note: {e}")

In [ ]:
import shutil

DRIVE_PATH = '/content/drive/MyDrive/kannada-legal-ai'
os.makedirs(DRIVE_PATH, exist_ok=True)

# Copy data folders to Drive so you don't have to re-run scraping
for folder in ['processed', 'annotated', 'vector_store']:
    local_dir = f'data/{folder}'
    drive_dir = f'{DRIVE_PATH}/data/{folder}'
    if os.path.exists(local_dir):
        shutil.copytree(local_dir, drive_dir, dirs_exist_ok=True)
        print(f'✅ Backed up data/{folder} to Google Drive')

print(f'\n🎉 Data collection complete! Backed up to: {DRIVE_PATH}')